In [ ]:
import pandas as pd
import smtplib
import sys
import time

from datetime import datetime, timedelta, timezone
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from google.colab import files, userdata
from random import randint

In [ ]:
uploaded = files.upload()

Saving Painel Global - Analisar e Assinar 07-08-2026 21_53.csv to Painel Global - Analisar e Assinar 07-08-2026 21_53.csv


In [ ]:
filename = list(uploaded.keys())[0]

In [ ]:
df = pd.read_csv(filename)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Classe              134 non-null    object
 1   Processo            134 non-null    object
 2   Fase                134 non-null    object
 3   Tarefa              134 non-null    object
 4   Desde               134 non-null    object
 5   Tipos de Atividade  73 non-null     object
 6   Responsável         134 non-null    object
 7   Prazo               63 non-null     object
dtypes: object(8)
memory usage: 8.5+ KB


In [ ]:
filter = df['Tarefa'] == 'Assinar acórdão'

In [ ]:
df_acordao = df[filter].copy()

In [ ]:
for pauta in df_acordao['Desde'].unique():
    print(pauta)

06/08/2026 22:21
06/08/2026 14:05
07/08/2026 13:07
07/08/2026 13:06
06/08/2026 14:26
06/08/2026 17:37
06/08/2026 14:49


In [ ]:
substituicoes = dict()
for data in df_acordao['Desde'].unique():
    pauta = input(f'Indique a pauta para {data}: ')
    substituicoes[data] = pauta

Indique a pauta para 06/08/2026 22:21: 
Indique a pauta para 06/08/2026 14:05: 
Indique a pauta para 07/08/2026 13:07: 
Indique a pauta para 07/08/2026 13:06: 
Indique a pauta para 06/08/2026 14:26: 
Indique a pauta para 06/08/2026 17:37: 
Indique a pauta para 06/08/2026 14:49: 


In [ ]:
substituicoes = {
    '06/08/2026 22:21': 'Pauta 13:05 (Sala com 98)',
    '06/08/2026 14:05': 'Pauta 13:50 (Sala com 62)',
    '07/08/2026 13:07': 'Pauta 13:05 (Sala com 98)',
    '07/08/2026 13:06': 'Pauta 13:05 (Sala com 98)',
    '06/08/2026 14:26': 'Pauta 13:02 (Sala com 4)',
    '06/08/2026 17:37': 'Pauta 13:03 (Sala com 76)',
    '06/08/2026 14:49': 'Pauta 13:01 (Sala com 18)',
}

In [ ]:
df_acordao.loc[:, 'Desde'] = df_acordao['Desde'].replace(substituicoes)

In [ ]:
def hora_atual():
    utc_now = datetime.now(timezone.utc)
    offset_brasilia = timedelta(hours=-3)
    hora_brasilia = utc_now + offset_brasilia
    return hora_brasilia.strftime('%H:%M:%S')

In [ ]:
def formatar_minutos_de_segundos(seconds):
    minutes = seconds // 60
    remaining_seconds = seconds % 60
    return f'{int(minutes)} minutos e {int(remaining_seconds)} segundos'

In [ ]:
def enviar_email(receiver_email, password, subject, html_body):
    sender_email = 'acordaos.informe@gmail.com'

    reply_to = 'gdhbl@trt12.jus.br'
    msg = MIMEMultipart()
    msg['From'] = f'Gab. Des. Helio Bastida Lopes <{sender_email}>'
    msg['Reply-To'] = reply_to
    msg['To'] = receiver_email
    msg['Bcc'] = sender_email
    msg['Subject'] = subject
    msg['Disposition-Notification-To'] = reply_to

    msg.attach(MIMEText(html_body, 'html'))

    try:
        with smtplib.SMTP('smtp.gmail.com', 587) as server:
            server.starttls()
            server.login(sender_email, password)
            server.send_message(msg)

        print(
            f'E-mail enviado com sucesso para {receiver_email} às {hora_atual()}'
        )
    except smtplib.SMTPAuthenticationError as e:
        raise smtplib.SMTPAuthenticationError(f'Erro de autenticação: {e}')
    except smtplib.SMTPConnectError as e:
        raise smtplib.SMTPConnectError(f'Erro de conexão: {e}')
    except smtplib.SMTPDataError as e:
        raise smtplib.SMTPDataError(
            f'Erro nos dados enviados ao servidor: {e}'
        )
    except Exception as e:
        raise Exception(f'Erro desconhecido ao enviar e-mail: {e}')

In [ ]:
destinatarios = {
    'ALEXANDRE MAIA DE MORAES': 'alexandre.moraes@trt12.jus.br',
    'CARLA LIMA': 'carla.lima@trt12.jus.br',
    'DUCLERQUE BEZERRA AGUIAR': 'duclerque.aguiar@trt12.jus.br',
    'EDUARDO MISSIO': 'eduardo.missio@trt12.jus.br',
    'ELISA REGINA FAVERO': 'elisa.favero@trt12.jus.br',
    'FELIPE SANTOS CAMARGOS': 'felipe.camargos@trt12.jus.br',
    'GRAZIELA TERESINHA VITORINO': 'graziela.vitorino@trt12.jus.br',
    'HEITOR MEDEIROS PERIN': 'heitor.perin@trt12.jus.br',
    'LUIZ PHILLIPE DE OLIVEIRA GOMES MARTINS': 'luiz.martins@trt12.jus.br',
    'GAB. DE APOIO': 'pedro.nora@trt12.jus.br',
    'PEDRO HENRIQUE MACEDO NORA': 'pedro.nora@trt12.jus.br',
    'TIAGO TEIXEIRA RODRIGUES': 'tiago.rodrigues@trt12.jus.br',
}

In [ ]:
df_acordao = df_acordao[['Classe', 'Processo', 'Responsável', 'Desde']].copy()
df_acordao.loc[:, 'Autos'] = (
    df_acordao['Classe'] + ' ' + df_acordao['Processo']
)

total_acordaos = int(input('Qual o total de acórdãos? '))
n_registros = df_acordao.shape[0]
if total_acordaos != n_registros:
    sys.exit(
        f'Inconsistência no número de acórdãos informados. Foram localizados {n_registros} registros e informados {total_acordaos}.'
    )

orgao = input('Informe o Órgão Colegiado: ')
data_sessao = input('Informe a data da sessão: ')
print()

responsaveis = sorted(df_acordao['Responsável'].unique())

total_enviados = 0

EMAIL_SENHA = userdata.get('EMAIL_SENHA')

for responsavel in responsaveis:
    filtro_responsavel = df_acordao['Responsável'] == responsavel
    resultado = (
        df_acordao[filtro_responsavel]
        .sort_values(by=['Desde', 'Processo'])
        .reset_index()
    )

    if resultado.empty:
        continue

    html_body = f"""
    <div style="text-align: center;">
        <div style="display: inline-block; text-align: left; width: 100%; max-width: 600px; margin-bottom: 5px;">Responsável: <strong>{responsavel}</strong></div>
        <table style="width: 100%; max-width: 600px; border-collapse: collapse; margin: 0 auto;">
            <thead>
                <tr>
                    <th style="padding: 10px; text-align: center; border: 1px solid #dddddd; background-color: #f2f2f2; color: black;">n.</th>
                    <th style="padding: 10px; text-align: center; border: 1px solid #dddddd; background-color: #f2f2f2; color: black;">PROCESSO</th>
                    <th style="padding: 10px; text-align: center; border: 1px solid #dddddd; background-color: #f2f2f2; color: black;">PAUTA</th>
                </tr>
            </thead>
            <tbody>
    """

    for index, row in resultado.iterrows():
        background_color = '#f9f9f9' if index % 2 == 0 else 'transparent'
        html_body += f"""
            <tr style="color: black; background-color: {background_color};">
                <td style="padding: 10px; text-align: center; border: 1px solid #dddddd;">{index + 1}</td>
                <td style="padding: 10px; text-align: center; border: 1px solid #dddddd;">{row['Autos']}</td>
                <td style="padding: 10px; text-align: center; border: 1px solid #dddddd;">{row['Desde']}</td>
            </tr>
        """

    html_body += """
            </tbody>
        </table>
    </div>
    """

    receiver_email = destinatarios.get(responsavel, 'pedro.nora@trt12.jus.br')
    assunto = f'[{orgao} - Sessão: {data_sessao}] Formatar acórdãos - {responsavel.split()[0].capitalize()}'

    try:
        enviar_email(receiver_email, EMAIL_SENHA, assunto, html_body)
        total_enviados += resultado.shape[0]
    except Exception as e:
        print(f'Erro ao enviar email para {responsavel}: {e}')

    print(
        f'Acórdãos enviados até o momento: {total_enviados}/{total_acordaos}'
    )

    if total_enviados < total_acordaos:
        seconds = randint(30, 60)
        print(f'Próximo email em {formatar_minutos_de_segundos(seconds)}.\n')
        time.sleep(seconds)

Qual o total de acórdãos? 134
Informe o Órgão Colegiado: 1ª Turma
Informe a data da sessão: 05/08/2026

E-mail enviado com sucesso para alexandre.moraes@trt12.jus.br às 22:20:48
Acórdãos enviados até o momento: 11/134
Próximo email em 0 minutos e 31 segundos.

E-mail enviado com sucesso para carla.lima@trt12.jus.br às 22:21:22
Acórdãos enviados até o momento: 23/134
Próximo email em 0 minutos e 33 segundos.

E-mail enviado com sucesso para duclerque.aguiar@trt12.jus.br às 22:21:58
Acórdãos enviados até o momento: 39/134
Próximo email em 0 minutos e 34 segundos.

E-mail enviado com sucesso para eduardo.missio@trt12.jus.br às 22:22:34
Acórdãos enviados até o momento: 45/134
Próximo email em 0 minutos e 48 segundos.

E-mail enviado com sucesso para elisa.favero@trt12.jus.br às 22:23:25
Acórdãos enviados até o momento: 55/134
Próximo email em 0 minutos e 51 segundos.

E-mail enviado com sucesso para felipe.camargos@trt12.jus.br às 22:24:19
Acórdãos enviados até o momento: 68/134
Próximo em